Here, we download the raw data from [McCauley 2024](https://doi.org/10.1016/j.cels.2024.02.005). 

Single-cell datasets include GEO: GSE246368.

In [1]:
import os
import zipfile

import scanpy as sc
import pandas as pd

import pooch

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

# Download Data

In [5]:
file_path = os.path.join(data_path, "raw")

Cell cycle data (as in [scanpy tutorial](https://scanpy.readthedocs.io/en/1.11.x/how-to/cell-cycle.html):

In [6]:
p_zip = pooch.retrieve(
    "https://www.dropbox.com/s/3dby3bjsaf5arrw/cell_cycle_vignette_files.zip?dl=1",
    known_hash="sha256:6557fe2a2761d1550d41edf61b26c6d5ec9b42b4933d56c3161164a595f145ab",
    path=file_path
)
with zipfile.ZipFile(p_zip, "r") as f_zip:
#     f_csv = zipfile.Path(f_zip, "nestorawa_forcellcycle_expressionMatrix.txt").open()
#     adata = sc.read_csv(f_csv, delimiter="\t").T
    cell_cycle_genes = zipfile.Path(f_zip, "regev_lab_cell_cycle_genes.txt").read_text().splitlines()

Download the HGNC database (10/22/25), needed for filtering the AnnData object for protein-coding genes:

In [7]:
fn = "HGNC_db.txt"

if not os.path.isfile(os.path.join(file_path, fn)):
    file_link = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/locus_types/gene_with_protein_product.txt"
    fn_0 = "gene_with_protein_product.txt"

    cmd = f"""
    wget -O "{file_path}/{fn_0}" "{file_link}" && \
    mv "{file_path}/{fn_0}" "{file_path}/{fn}"
    """

    os.system(cmd)

Download raw counts of GSE246368 from [Zenodo](https://zenodo.org/records/10602177). This is the same as 'GSE246368_all_data.h5ad.gz' from [GSE246368](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE246368) but with more information (e.g. UMAP). 

Since the data associated with GSE246441 (additional donors for control, OSM, and IFN) was only used by the paper for Fig 1 abundance plots (according to where it was used in the [Github](https://github.com/AllonKleinLab/paper-data/tree/master/McCauley_Kukreja_2024/notebooks)), we will disregard it.

In [8]:
fn = author + "_GSE246368.h5ad"

# GEO version
# # wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE246nnn/GSE246368/suppl/GSE246368%5Fall%5Fdata.h5ad.gz
# # gunzip GSE246368_all_data.h5ad.gz
# # mv GSE246368_all_data.h5ad McCauley_GSE246368.h5ad

# if not os.path.isfile(os.path.join(file_path, fn)):
#     file_link = 'https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1'
#     fn_0 = "all_indrops_data.h5ad.zip?download=1"
#     fn_1 = 'all_indrops_data.h5ad '

#     cmd = f"""
#     wget -O "{file_path}/{fn_0}.gz" "{file_link}" && \
#     gunzip -f "{file_path}/{fn_0}.gz" && \
#     mv "{file_path}/{fn_1}" "{file_path}/{fn}"
#     """

#     os.system(cmd)


# Zenodo version
# wget https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1
# unzip 'all_indrops_data.h5ad.zip?download=1'
# rm 'all_indrops_data.h5ad.zip?download=1'
# mv all_indrops_data.h5ad McCauley_GSE246368.h5ad

if not os.path.isfile(os.path.join(file_path, fn)):
    file_link = "https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1"
    fn_0 = "all_indrops_data.h5ad.zip?download=1"

    cmd = f"""
    wget -O "{file_path}/{fn_0}" "{file_link}" && \
    unzip "{file_path}/{fn_0}" -d "{file_path}" && \
    rm "{file_path}/{fn_0}" && \
    mv "{file_path}/all_indrops_data.h5ad" "{file_path}/{fn}"
    """

    os.system(cmd)

# Preprocessing

In [9]:
file_path = os.path.join(data_path, "raw")
fn = author + "_GSE246368.h5ad"
adata = sc.read_h5ad(os.path.join(file_path, fn))


In [10]:
# filter out doublets and unassigned cells
adata = adata[~adata.obs.assigned_donor.isin(['doublet', 'Unassigned'])] 

# retain only feaetures that are protein coding genes
hgnc_genes = pd.read_csv(os.path.join(data_path, 'raw', 'HGNC_db.txt'), 
                        sep = '\t', low_memory=False).symbol.tolist()
gene_mask = [True if vn in hgnc_genes else False for vn in adata.var_names]
adata = adata[:, gene_mask]

# basic QC -- same thresholds as in original paper (https://github.com/AllonKleinLab/paper-data/blob/master/McCauley_Kukreja_2024/notebooks/fig1_2_analyses/control_neighbors_analyses_fig1.ipynb)
sc.pp.filter_genes(adata, min_counts=6)
sc.pp.filter_genes(adata, min_cells=3)

# normalize
adata.layers["counts"] = adata.X.copy() # store the raw data
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)

# hvgs
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key=None, flavor = 'seurat')


# cell cycle scoring
# scanpy tutorial(https://scanpy.readthedocs.io/en/1.11.x/how-to/cell-cycle.html)
s_genes = [x for x in cell_cycle_genes[:43] if x in adata.var_names]
g2m_genes = [x for x in cell_cycle_genes[43:] if x in adata.var_names]
cell_cycle_genes = [*s_genes, *g2m_genes]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:291: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_counts"] = number


# Latent Space Quantification

To do: this if proceeding with the dataset

In [11]:
adata.write_h5ad(os.path.join(data_path, 'processed', author + '_normalized_counts.h5ad'))